# 03 – Embedding + MLP Architecture

## 1. Introduction

This notebook focuses on designing and implementing a Multi-Layer Perceptron (MLP) architecture with learned categorical embeddings for flight delay prediction.

The objective is to develop a neural network that combines numerical and categorical flight features to predict whether a flight will experience a substantial arrival delay.

The architecture uses separate embedding layers for categorical variables and combines their learned representations with numerical features.

The combined feature representation is passed through fully connected layers to generate a binary classification logit.

### Key Components

- PyTorch embedding layers for categorical features.
- Numerical feature input.
- Feature concatenation.
- Fully connected neural network layers.
- ReLU activation functions.
- Dropout regularisation.
- Forward propagation.

### Input Features

**Categorical Features:**
- carrier
- origin
- dest
- route

**Numerical Features:**
- distance
- dep_time_sin
- dep_time_cos
- weekday_sin
- weekday_cos
- month_sin
- month_cos
- arr_time_sin
- arr_time_cos
- origin_delay_rate_prev7d
- origin_flights_prev7d
- carrier_delay_rate_prev7d
- carrier_flights_prev7d
- dest_delay_rate_prev7d
- dest_flights_prev7d
- route_delay_rate_prev7d
- route_flights_prev7d

**Target:** `delayed`

The model architecture will be designed independently of the training and evaluation pipeline, which will be implemented in subsequent notebooks.

## 2. Importing Required Libraries

We will use PyTorch to implement the neural network architecture.

The `torch.nn` module provides the building blocks required to construct the model, including embedding layers, fully connected layers, activation functions and dropout.

NumPy and Matplotlib will be used for numerical operations and architecture-related visualisation where required.

In [1]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

import json
import os

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

PyTorch Version: 2.14.1+cpu
CUDA Available: False


## 3. Load Preprocessed Data

This section loads the preprocessed dataset generated in Notebook 02.

The MLP requires:
- Numerical feature tensors.
- Categorical feature tensors.
- Categorical mappings to determine the number of embeddings required for each categorical feature.

No additional preprocessing or data splitting is performed in this notebook.

In [2]:
from google.colab import files

uploaded = files.upload()

Saving dl_data.pt to dl_data.pt


In [3]:
data = torch.load(
    "dl_data.pt",
    map_location="cpu",
    weights_only=False
)

print("Available keys:")
for key in data.keys():
    print(key)

Available keys:
X_train_num
X_train_cat
y_train
X_val_num
X_val_cat
y_val
X_test_num
X_test_cat
y_test
numerical_features
categorical_features
category_maps
train_medians
scaler_mean
scaler_std


In [4]:
X_train_num = data["X_train_num"]
X_train_cat = data["X_train_cat"]

numerical_features = data["numerical_features"]
categorical_features = data["categorical_features"]
category_maps = data["category_maps"]

print("Numerical input shape:", X_train_num.shape)
print("Categorical input shape:", X_train_cat.shape)

print("\nNumerical features:", numerical_features)
print("\nCategorical features:", categorical_features)

Numerical input shape: torch.Size([217727, 17])
Categorical input shape: torch.Size([217727, 4])

Numerical features: ['distance', 'dep_time_sin', 'dep_time_cos', 'weekday_sin', 'weekday_cos', 'month_sin', 'month_cos', 'arr_time_sin', 'arr_time_cos', 'origin_delay_rate_prev7d', 'origin_flights_prev7d', 'carrier_delay_rate_prev7d', 'carrier_flights_prev7d', 'dest_delay_rate_prev7d', 'dest_flights_prev7d', 'route_delay_rate_prev7d', 'route_flights_prev7d']

Categorical features: ['carrier', 'origin', 'dest', 'route']


## 4. Understanding Categorical Embeddings

Categorical variables represent discrete groups rather than continuous numerical quantities.

For example:
- `carrier` represents an airline.
- `origin` represents the departure airport.
- `dest` represents the arrival airport.
- `route` represents a particular flight route.

Instead of treating category IDs as meaningful numerical values, we use PyTorch's `nn.Embedding` layer.

An embedding layer maps each category ID to a trainable dense vector. These vectors are learned during model training.

For each categorical feature, we create a separate embedding layer. The resulting embedding vectors are concatenated with the numerical features before being passed into the fully connected network.

The embedding dimensions are selected using a simple heuristic:

\[
d = \min(50,\lceil \frac{N}{2} \rceil)
\]

where:
- \(N\) is the number of unique categories.
- \(d\) is the embedding dimension.

This provides a reasonable starting architecture. The training team can later experiment with different dimensions if required.

In [5]:
category_sizes = {}

print("Categorical Feature Information\n")

for feature in categorical_features:
    mapping = category_maps[feature]
    indices = list(mapping.values())

    category_count = max(indices) + 1

    category_sizes[feature] = category_count

    print(
        f"{feature}: "
        f"{category_count} categories | "
        f"Index range: {min(indices)} to {max(indices)}"
    )

Categorical Feature Information

carrier: 17 categories | Index range: 0 to 16
origin: 4 categories | Index range: 0 to 3
dest: 103 categories | Index range: 0 to 102
route: 219 categories | Index range: 0 to 218


In [6]:
embedding_dims = {}

for feature in categorical_features:
    num_categories = category_sizes[feature]

    embedding_dim = min(
        50,
        int(np.ceil(num_categories / 2))
    )

    embedding_dims[feature] = embedding_dim

print("Embedding Configuration:\n")

for feature in categorical_features:
    print(
        f"{feature}: "
        f"{category_sizes[feature]} categories -> "
        f"{embedding_dims[feature]} dimensions"
    )

Embedding Configuration:

carrier: 17 categories -> 9 dimensions
origin: 4 categories -> 2 dimensions
dest: 103 categories -> 50 dimensions
route: 219 categories -> 50 dimensions


In [7]:
assert X_train_num.ndim == 2, "Numerical input must be 2-dimensional"
assert X_train_cat.ndim == 2, "Categorical input must be 2-dimensional"

assert X_train_num.shape[1] == len(numerical_features)
assert X_train_cat.shape[1] == len(categorical_features)

assert len(category_maps) == len(categorical_features)

for feature in categorical_features:
    indices = sorted(category_maps[feature].values())

    assert indices == list(range(len(indices))), (
        f"Category indices for {feature} are not contiguous from zero."
    )

print("Input dimensions and category mappings verified.")

Input dimensions and category mappings verified.


In [8]:
print("Numerical tensor dtype:", X_train_num.dtype)
print("Categorical tensor dtype:", X_train_cat.dtype)

for i, feature in enumerate(categorical_features):
    min_index = X_train_cat[:, i].min().item()
    max_index = X_train_cat[:, i].max().item()

    print(
        f"{feature}: minimum index = {min_index}, "
        f"maximum index = {max_index}, "
        f"embedding size = {category_sizes[feature]}"
    )

    assert min_index >= 0
    assert max_index < category_sizes[feature]

assert torch.is_floating_point(X_train_num)
assert X_train_cat.dtype == torch.long

print("\nAll input tensors are compatible with the embedding model.")

Numerical tensor dtype: torch.float32
Categorical tensor dtype: torch.int64
carrier: minimum index = 1, maximum index = 16, embedding size = 17
origin: minimum index = 1, maximum index = 3, embedding size = 4
dest: minimum index = 1, maximum index = 102, embedding size = 103
route: minimum index = 1, maximum index = 218, embedding size = 219

All input tensors are compatible with the embedding model.


## 5. MLP Architecture Design

The model consists of two input pathways.

**Pathway 1 — Categorical Features**

Each categorical feature is passed through its own embedding layer:

- Carrier Embedding
- Origin Embedding
- Destination Embedding
- Route Embedding

The four embedding outputs are concatenated into a single categorical representation.

**Pathway 2 — Numerical Features**

The 17 numerical features are passed directly into the model.

**Feature Fusion**

The numerical features and categorical embeddings are concatenated into one feature vector.

**Fully Connected Network**

The combined representation passes through three fully connected layers:

1. Linear layer with 128 neurons, followed by ReLU and Dropout.
2. Linear layer with 64 neurons, followed by ReLU and Dropout.
3. Linear layer with 32 neurons, followed by ReLU.

**Output Layer**

A final linear layer produces one raw logit representing the predicted likelihood of a substantial arrival delay.

The model returns logits rather than applying a sigmoid activation. This allows the training component to use an appropriate logits-based loss function.

## 6. Implementing the Embedding MLP

We implement the architecture using a custom PyTorch `nn.Module` class.

The model accepts:
- A batch of numerical features.
- A batch of categorical feature indices.

Separate embedding layers are created for the four categorical variables. Their outputs are concatenated with the numerical inputs.

The combined representation passes through fully connected layers with ReLU activations and dropout regularization.

The hidden layer sizes and dropout rates below represent the initial architecture. They are configurable so the training team can experiment with them later.

In [9]:
class FlightDelayMLP(nn.Module):

    def __init__(
        self,
        numerical_input_dim,
        category_sizes,
        embedding_dims,
        hidden_dims=(128, 64, 32),
        dropout_rates=(0.3, 0.2)
    ):
        super().__init__()

        self.num_categorical_features = len(category_sizes)

        self.embeddings = nn.ModuleList([
            nn.Embedding(
                num_embeddings=category_sizes[feature],
                embedding_dim=embedding_dims[feature]
            )
            for feature in category_sizes
        ])

        total_embedding_dim = sum(embedding_dims.values())

        combined_input_dim = (
            numerical_input_dim + total_embedding_dim
        )

        layers = []

        previous_dim = combined_input_dim

        for i, hidden_dim in enumerate(hidden_dims):

            layers.append(
                nn.Linear(previous_dim, hidden_dim)
            )

            layers.append(nn.ReLU())

            if i < len(dropout_rates):
                layers.append(
                    nn.Dropout(dropout_rates[i])
                )

            previous_dim = hidden_dim

        self.mlp = nn.Sequential(*layers)

        self.output_layer = nn.Linear(
            previous_dim,
            1
        )

    def forward(self, x_num, x_cat):

        embedded_features = []

        for i, embedding_layer in enumerate(self.embeddings):

            embedded = embedding_layer(x_cat[:, i])

            embedded_features.append(embedded)

        categorical_output = torch.cat(
            embedded_features,
            dim=1
        )

        combined_input = torch.cat(
            [x_num, categorical_output],
            dim=1
        )

        hidden_output = self.mlp(combined_input)

        logits = self.output_layer(hidden_output)

        return logits.squeeze(1)

## 7. Initialize the MLP Model

We initialize the model using the numerical feature count, categorical cardinalities, and embedding dimensions derived from the preprocessing output.

The model is initialized with its default random weights. No training is performed in this notebook.

In [10]:
model = FlightDelayMLP(
    numerical_input_dim=len(numerical_features),
    category_sizes=category_sizes,
    embedding_dims=embedding_dims,
    hidden_dims=(128, 64, 32),
    dropout_rates=(0.3, 0.2)
)

print(model)

FlightDelayMLP(
  (embeddings): ModuleList(
    (0): Embedding(17, 9)
    (1): Embedding(4, 2)
    (2): Embedding(103, 50)
    (3): Embedding(219, 50)
  )
  (mlp): Sequential(
    (0): Linear(in_features=128, out_features=128, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=128, out_features=64, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.2, inplace=False)
    (6): Linear(in_features=64, out_features=32, bias=True)
    (7): ReLU()
  )
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
)


## 8. Forward Pass and Dimension Testing

The forward pass checks whether numerical and categorical inputs are correctly processed by the embedding layers and fully connected network.

We use a small batch from the training input tensors only to verify the model's input and output dimensions.

This is an architecture test, not a training or evaluation step.

In [11]:
batch_size = 8

sample_num = X_train_num[:batch_size]
sample_cat = X_train_cat[:batch_size]

model.eval()

with torch.no_grad():
    sample_output = model(
        sample_num,
        sample_cat
    )

print("Numerical input shape:", sample_num.shape)
print("Categorical input shape:", sample_cat.shape)
print("Model output shape:", sample_output.shape)

assert sample_output.shape == (batch_size,)

print("\nForward pass successful!")

Numerical input shape: torch.Size([8, 17])
Categorical input shape: torch.Size([8, 4])
Model output shape: torch.Size([8])

Forward pass successful!


## 9. Model Parameters

We calculate the total number of trainable parameters in the model.

Trainable parameters include:
- Embedding weights.
- Fully connected layer weights and biases.
- Final prediction layer weights and biases.

The parameter count helps describe the model's complexity and provides useful information for the later AML-versus-DL comparison.

In [12]:
total_parameters = sum(
    parameter.numel()
    for parameter in model.parameters()
    if parameter.requires_grad
)

embedding_parameters = sum(
    parameter.numel()
    for embedding in model.embeddings
    for parameter in embedding.parameters()
)

mlp_parameters = (
    sum(
        parameter.numel()
        for parameter in model.mlp.parameters()
    )
    +
    sum(
        parameter.numel()
        for parameter in model.output_layer.parameters()
    )
)

print("MODEL PARAMETER SUMMARY")
print("-----------------------")

print("Embedding parameters:", f"{embedding_parameters:,}")
print("Fully connected parameters:", f"{mlp_parameters:,}")
print("Total trainable parameters:", f"{total_parameters:,}")

assert total_parameters == embedding_parameters + mlp_parameters

MODEL PARAMETER SUMMARY
-----------------------
Embedding parameters: 16,261
Fully connected parameters: 26,881
Total trainable parameters: 43,142


## 10. Architecture Configuration

This section records the architecture configuration used in this notebook.

The configuration includes:
- Numerical input dimensions.
- Categorical feature names and category counts.
- Embedding dimensions.
- Hidden layer dimensions.
- Dropout rates.
- Total trainable parameters.

This configuration documents the initial architecture and can be used by the training team when integrating the model into Notebook 04.

In [13]:
model_config = {
    "model_name": "FlightDelayMLP",

    "numerical_features": numerical_features,

    "categorical_features": categorical_features,

    "category_sizes": category_sizes,

    "embedding_dimensions": embedding_dims,

    "numerical_input_dim": len(numerical_features),

    "hidden_dimensions": [128, 64, 32],

    "dropout_rates": [0.3, 0.2],

    "output_dimension": 1,

    "output_type": "raw_logit",

    "total_trainable_parameters": total_parameters
}

print(json.dumps(model_config, indent=4))

{
    "model_name": "FlightDelayMLP",
    "numerical_features": [
        "distance",
        "dep_time_sin",
        "dep_time_cos",
        "weekday_sin",
        "weekday_cos",
        "month_sin",
        "month_cos",
        "arr_time_sin",
        "arr_time_cos",
        "origin_delay_rate_prev7d",
        "origin_flights_prev7d",
        "carrier_delay_rate_prev7d",
        "carrier_flights_prev7d",
        "dest_delay_rate_prev7d",
        "dest_flights_prev7d",
        "route_delay_rate_prev7d",
        "route_flights_prev7d"
    ],
    "categorical_features": [
        "carrier",
        "origin",
        "dest",
        "route"
    ],
    "category_sizes": {
        "carrier": 17,
        "origin": 4,
        "dest": 103,
        "route": 219
    },
    "embedding_dimensions": {
        "carrier": 9,
        "origin": 2,
        "dest": 50,
        "route": 50
    },
    "numerical_input_dim": 17,
    "hidden_dimensions": [
        128,
        64,
        32
    ],
    "dro

In [14]:
with open("model_config.json", "w") as file:
    json.dump(model_config, file, indent=4)

print("Model configuration saved successfully.")

Model configuration saved successfully.


In [15]:
print("=" * 50)
print("FLIGHT DELAY MLP - ARCHITECTURE VERIFICATION")
print("=" * 50)

print("Numerical features:", len(numerical_features))
print("Categorical features:", len(categorical_features))
print("Embedding dimensions:", embedding_dims)
print("Combined input dimensions:", len(numerical_features) + sum(embedding_dims.values()))
print("Hidden layers:", [128, 64, 32])
print("Dropout rates:", [0.3, 0.2])
print("Output dimension: 1")
print("Output type: Raw logit")
print("Trainable parameters:", f"{total_parameters:,}")

print("\nForward pass test:", "PASSED" if sample_output.shape == (batch_size,) else "FAILED")

print("\nArchitecture implementation completed.")

FLIGHT DELAY MLP - ARCHITECTURE VERIFICATION
Numerical features: 17
Categorical features: 4
Embedding dimensions: {'carrier': 9, 'origin': 2, 'dest': 50, 'route': 50}
Combined input dimensions: 128
Hidden layers: [128, 64, 32]
Dropout rates: [0.3, 0.2]
Output dimension: 1
Output type: Raw logit
Trainable parameters: 43,142

Forward pass test: PASSED

Architecture implementation completed.
